# Импорт данных с экономией памяти

## Принципы импорта

Опираемся на текущую документацию и на фактические CSV.

Что делаем для экономии памяти:
- не читаем `Unnamed: 0`;
- пропускаем поля, которые в документации помечены как ненужные, а также тяжёлые детали вроде `results`;
- читаем числа с `thousands=","`, потому что ID в CSV записаны с разделителями тысяч;
- сразу переводим low-cardinality строки в `category`;
- downcast-им числовые столбцы до `Int8/Int16/Int32` и `Float32`, где это безопасно.


In [6]:
from pathlib import Path

import pandas as pd

SRC_DIR = Path.cwd() / "src"


In [7]:
# Базовые таблицы: пользователи, курсы и структура курса.
users_df = pd.read_csv(
    SRC_DIR / "users.csv",
    usecols=[
        "id",
        "last_explainer_seen_→_course",
        "created_at",
        "updated_at",
        "type",
        "sign_in_count",
        "grade_id",
        "subscribed",
        "is_teacher",
        "timezone",
        "grade_changed_at",
        "d_wk_school_id",
        "d_wk_municipal_id",
        "d_wk_region_id",
        "wk_gender",
    ],
    thousands=",",
    low_memory=False,
)
users_df["created_at"] = pd.to_datetime(users_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["updated_at"] = pd.to_datetime(users_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["grade_changed_at"] = pd.to_datetime(users_df["grade_changed_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["id"] = users_df["id"].astype("Int32")
users_df["last_explainer_seen_→_course"] = users_df["last_explainer_seen_→_course"].astype("Int8")
users_df["sign_in_count"] = users_df["sign_in_count"].astype("Int32")
users_df["grade_id"] = users_df["grade_id"].astype("Int16")
users_df["d_wk_school_id"] = users_df["d_wk_school_id"].astype("Int32")
users_df["d_wk_municipal_id"] = users_df["d_wk_municipal_id"].astype("Int32")
users_df["d_wk_region_id"] = users_df["d_wk_region_id"].astype("Int32")
users_df["wk_gender"] = users_df["wk_gender"].astype("Int8")
users_df["type"] = users_df["type"].astype("category")
users_df["timezone"] = users_df["timezone"].astype("category")
users_df["subscribed"] = users_df["subscribed"].astype("boolean")
users_df["is_teacher"] = users_df["is_teacher"].astype("boolean")

users_courses_df = pd.read_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "user_id",
        "course_id",
        "state",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_viewable_lessons",
        "wk_max_task_count",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    thousands=",",
    low_memory=False,
)
users_courses_df["created_at"] = pd.to_datetime(users_courses_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["updated_at"] = pd.to_datetime(users_courses_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["access_finished_at"] = pd.to_datetime(users_courses_df["access_finished_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_officially_started_at"] = pd.to_datetime(users_courses_df["wk_officially_started_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_course_completed_at"] = pd.to_datetime(users_courses_df["wk_course_completed_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["user_id"] = users_courses_df["user_id"].astype("Int32")
users_courses_df["course_id"] = users_courses_df["course_id"].astype("Int32")
users_courses_df["wk_points"] = users_courses_df["wk_points"].astype("Float32")
users_courses_df["wk_max_points"] = users_courses_df["wk_max_points"].astype("Float32")
users_courses_df["wk_max_viewable_lessons"] = users_courses_df["wk_max_viewable_lessons"].astype("Float32")
users_courses_df["wk_max_task_count"] = users_courses_df["wk_max_task_count"].astype("Float32")
users_courses_df["state"] = users_courses_df["state"].astype("category")

lessons_df = pd.read_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "course_id",
        "conspect_expected",
        "task_expected",
        "lesson_number",
        "wk_max_points",
        "wk_task_count",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
        "wk_video_duration",
        "wk_attendance_tracking_disabled_at",
    ],
    parse_dates=["wk_attendance_tracking_disabled_at"],
    thousands=",",
    low_memory=False,
)
lessons_df["course_id"] = lessons_df["course_id"].astype("Int32")
lessons_df["lesson_number"] = lessons_df["lesson_number"].astype("Float32")
lessons_df["wk_max_points"] = lessons_df["wk_max_points"].astype("Float32")
lessons_df["wk_task_count"] = lessons_df["wk_task_count"].astype("Float32")
lessons_df["wk_video_duration"] = lessons_df["wk_video_duration"].astype("Float32")

user_access_histories_df = pd.read_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    thousands=",",
    low_memory=False,
)
user_access_histories_df["access_started_at"] = pd.to_datetime(user_access_histories_df["access_started_at"], format="%d %b, %Y", errors="coerce")
user_access_histories_df["access_expired_at"] = pd.to_datetime(user_access_histories_df["access_expired_at"], format="%d %b, %Y", errors="coerce")
user_access_histories_df["users_course_id"] = user_access_histories_df["users_course_id"].astype("Int32")
user_access_histories_df["activator_class"] = user_access_histories_df["activator_class"].astype("category")


In [8]:
# Логи прохождения курса.
user_lessons_df = pd.read_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "video_viewed",
        "wk_solved_task_count",
    ],
    thousands=",",
    low_memory=False,
)
user_lessons_df["user_id"] = user_lessons_df["user_id"].astype("Int32")
user_lessons_df["lesson_id"] = user_lessons_df["lesson_id"].astype("Int32")
user_lessons_df["group_id"] = user_lessons_df["group_id"].astype("Int32")
user_lessons_df["users_course_id"] = user_lessons_df["users_course_id"].astype("Int32")
user_lessons_df["solved_tasks_count"] = user_lessons_df["solved_tasks_count"].astype("Int16")
user_lessons_df["wk_points"] = user_lessons_df["wk_points"].astype("Float32")
user_lessons_df["wk_solved_task_count"] = user_lessons_df["wk_solved_task_count"].astype("Float32")
user_lessons_df[["video_visited", "translation_visited", "solved", "video_viewed"]] = user_lessons_df[["video_visited", "translation_visited", "solved", "video_viewed"]].astype("boolean")

user_activity_histories_df = pd.read_csv(
    SRC_DIR / "user_activity_histories.csv",
    usecols=["user_lesson_id", "action", "created_at"],
    parse_dates=["created_at"],
    thousands=",",
    low_memory=False,
)
user_activity_histories_df["user_lesson_id"] = user_activity_histories_df["user_lesson_id"].astype("Int32")
user_activity_histories_df["action"] = user_activity_histories_df["action"].astype("category")

wk_users_courses_actions_df = pd.read_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    parse_dates=["created_at", "updated_at"],
    thousands=",",
    low_memory=False,
)
wk_users_courses_actions_df["user_id"] = wk_users_courses_actions_df["user_id"].astype("Int32")
wk_users_courses_actions_df["users_course_id"] = wk_users_courses_actions_df["users_course_id"].astype("Int32")
wk_users_courses_actions_df["lesson_id"] = wk_users_courses_actions_df["lesson_id"].astype("Int32")
wk_users_courses_actions_df["action"] = wk_users_courses_actions_df["action"].astype("category")


In [ ]:
# Ответы, тренинги, бейджи и просмотр медиа.
user_answers_df = pd.read_csv(
    SRC_DIR / "user_answers.csv",
    usecols=[
        "user_id",
        "task_id",
        # "attempts", # константная колонка, не загружаем
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    parse_dates=["submitted_at"],
    thousands=",",
    low_memory=False,
)
user_answers_df["user_id"] = user_answers_df["user_id"].astype("Int32")
user_answers_df["task_id"] = user_answers_df["task_id"].astype("Int32")
user_answers_df["attempts"] = user_answers_df["attempts"].astype("Int8")
user_answers_df["points"] = user_answers_df["points"].astype("Float32")
user_answers_df["max_attempts"] = user_answers_df["max_attempts"].astype("Int8")
user_answers_df["async_check_status"] = user_answers_df["async_check_status"].astype("Int8")
user_answers_df["resource_type"] = user_answers_df["resource_type"].astype("category")
user_answers_df["solved"] = user_answers_df["solved"].replace({"True": True, "False": False}).astype("boolean")
user_answers_df["skipped"] = user_answers_df["skipped"].replace({"True": True, "False": False}).astype("boolean")
user_answers_df["wk_partial_answer"] = user_answers_df["wk_partial_answer"].replace({"True": True, "False": False}).astype("boolean")

user_trainings_df = pd.read_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=[
        "user_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "type",
        "state",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
        "mark_saved_at",
    ],
    thousands=",",
    low_memory=False,
)
user_trainings_df["started_at"] = pd.to_datetime(user_trainings_df["started_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["finished_at"] = pd.to_datetime(user_trainings_df["finished_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["mark_saved_at"] = pd.to_datetime(user_trainings_df["mark_saved_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["user_id"] = user_trainings_df["user_id"].astype("Int32")
user_trainings_df["training_id"] = user_trainings_df["training_id"].astype("Int32")
user_trainings_df["solved_tasks_count"] = user_trainings_df["solved_tasks_count"].astype("Int16")
user_trainings_df["earned_points"] = user_trainings_df["earned_points"].astype("Float32")
user_trainings_df["submitted_answers_count"] = user_trainings_df["submitted_answers_count"].astype("Int16")
user_trainings_df["attempts"] = user_trainings_df["attempts"].astype("Int8")
user_trainings_df["mark"] = user_trainings_df["mark"].astype("Float32")
user_trainings_df["type"] = user_trainings_df["type"].astype("category")
user_trainings_df["state"] = user_trainings_df["state"].astype("category")

user_award_badges_df = pd.read_csv(
    SRC_DIR / "user_award_badges.csv",
    usecols=["award_badge_id", "user_id", "created_at"],
    parse_dates=["created_at"],
    thousands=",",
    low_memory=False,
)
user_award_badges_df["award_badge_id"] = user_award_badges_df["award_badge_id"].astype("Int8")
user_award_badges_df["user_id"] = user_award_badges_df["user_id"].astype("Int32")

award_badges_df = pd.read_csv(
    SRC_DIR / "award_badges.csv",
    usecols=["title", "level", "quota", "special"],
    low_memory=False,
)
award_badges_df["title"] = award_badges_df["title"].astype("category")
award_badges_df["level"] = award_badges_df["level"].astype("Int8")
award_badges_df["quota"] = award_badges_df["quota"].astype("Int16")
award_badges_df["special"] = award_badges_df["special"].astype("boolean")

wk_media_view_sessions_df = pd.read_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    thousands=",",
    low_memory=False,
)
wk_media_view_sessions_df["started_at"] = pd.to_datetime(wk_media_view_sessions_df["started_at"], format="%d %b, %Y, %H:%M", errors="coerce")
wk_media_view_sessions_df["resource_type"] = wk_media_view_sessions_df["resource_type"].astype("category")
wk_media_view_sessions_df["viewer_id"] = wk_media_view_sessions_df["viewer_id"].astype("Int32")
wk_media_view_sessions_df["segments_total"] = wk_media_view_sessions_df["segments_total"].astype("Int16")
wk_media_view_sessions_df["viewed_segments_count"] = wk_media_view_sessions_df["viewed_segments_count"].astype("Int16")
wk_media_view_sessions_df["kind"] = wk_media_view_sessions_df["kind"].astype("category")


In [10]:
dataframes = {
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "award_badges_df": award_badges_df,
    "user_award_badges_df": user_award_badges_df,
}

import_summary_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
        "memory_mb": [round(df.memory_usage(deep=True).sum() / 1024**2, 2) for df in dataframes.values()],
    }
).sort_values("memory_mb", ascending=False, ignore_index=True)

total_memory_mb = round(import_summary_df["memory_mb"].sum(), 2)

display(import_summary_df)
display(f"Суммарно в памяти: {total_memory_mb} MB")


,dataframe,rows,cols,memory_mb
0,user_answers_df,15176182,11,521.03
1,wk_users_courses_actions_df,12909207,6,393.96
2,user_lessons_df,3070664,11,120.07
3,user_activity_histories_df,3031137,3,40.47
4,user_trainings_df,427628,12,22.02
5,users_courses_df,290835,12,19.69
6,wk_media_view_sessions_df,852358,6,17.07
7,user_access_histories_df,667124,4,14.00
8,users_df,95395,15,5.75
9,user_award_badges_df,252843,3,3.62


'Суммарно в памяти: 1157.8 MB'

## Аудит качества данных

Для первого прохода не усложняем и смотрим по каждой таблице:
- размер;
- `info()`;
- `describe()`.

После этого отдельно делаем несколько простых sanity checks по датам и ограничениям значений.


In [11]:
display(import_summary_df)

for name, df in dataframes.items():
    print(f"\n{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    df.info(show_counts=True)
    display(df.describe(include="all").T)


,dataframe,rows,cols,memory_mb
0,user_answers_df,15176182,11,521.03
1,wk_users_courses_actions_df,12909207,6,393.96
2,user_lessons_df,3070664,11,120.07
3,user_activity_histories_df,3031137,3,40.47
4,user_trainings_df,427628,12,22.02
5,users_courses_df,290835,12,19.69
6,wk_media_view_sessions_df,852358,6,17.07
7,user_access_histories_df,667124,4,14.00
8,users_df,95395,15,5.75
9,user_award_badges_df,252843,3,3.62



users_df: 95,395 rows x 15 cols
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95395 entries, 0 to 95394
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id                            95395 non-null  Int32         
 1   last_explainer_seen_→_course  8217 non-null   Int8          
 2   created_at                    95395 non-null  datetime64[ns]
 3   updated_at                    95395 non-null  datetime64[ns]
 4   type                          95395 non-null  category      
 5   sign_in_count                 95395 non-null  Int32         
 6   grade_id                      95395 non-null  Int16         
 7   subscribed                    95395 non-null  boolean       
 8   is_teacher                    95395 non-null  boolean       
 9   timezone                      95217 non-null  category      
 10  grade_changed_at              1804 non-null   datetime64[ns]


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,95395.0,<NA>,<NA>,<NA>,713619.057257,665740.0,689652.5,713630.0,737571.5,761603.0,27654.314533
last_explainer_seen_→_course,8217.0,<NA>,<NA>,<NA>,6.382743,1.0,7.0,7.0,7.0,7.0,1.449124
created_at,95395,NaN,NaN,NaN,2025-09-02 10:04:45.591068928,2025-01-31 14:16:00,2025-06-05 19:58:30,2025-09-30 11:31:00,2025-11-12 08:38:00,2026-03-27 16:13:00,NaN
updated_at,95395,NaN,NaN,NaN,2025-11-25 07:39:32.594789888,2025-02-17 07:30:00,2025-11-18 20:30:30,2025-12-16 12:16:00,2025-12-25 01:55:00,2026-03-27 16:47:00,NaN
type,95395,2,User::Pupil,90647,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sign_in_count,95395.0,<NA>,<NA>,<NA>,18.956077,0.0,4.0,9.0,19.0,1390.0,37.564549
grade_id,95395.0,<NA>,<NA>,<NA>,3009.521076,3000.0,3010.0,3010.0,3010.0,3012.0,1.935844
subscribed,95395,2,True,94940,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_teacher,95395,1,False,95395,NaN,NaN,NaN,NaN,NaN,NaN,NaN
timezone,95217,141,Europe/Moscow,64085,NaN,NaN,NaN,NaN,NaN,NaN,NaN



users_courses_df: 290,835 rows x 12 cols
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290835 entries, 0 to 290834
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   user_id                   290835 non-null  Int32         
 1   course_id                 290835 non-null  Int32         
 2   state                     290835 non-null  category      
 3   created_at                290835 non-null  datetime64[ns]
 4   updated_at                290835 non-null  datetime64[ns]
 5   access_finished_at        290741 non-null  datetime64[ns]
 6   wk_points                 205925 non-null  Float32       
 7   wk_max_points             290710 non-null  Float32       
 8   wk_max_viewable_lessons   290710 non-null  Float32       
 9   wk_max_task_count         290710 non-null  Float32       
 10  wk_officially_started_at  96865 non-null   datetime64[ns]
 11  wk_course_completed_at 

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,290835.0,<NA>,<NA>,<NA>,715722.751409,665740.0,692031.0,716070.0,739958.0,761578.0,27293.035063
course_id,290835.0,<NA>,<NA>,<NA>,6916813.866577,754.0,770.0,771.0,836.0,170000688.0,29739455.750645
state,290835,2,active,287548,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,290835,NaN,NaN,NaN,2025-10-02 12:20:35.549848064,2025-02-07 11:33:00,2025-07-31 00:33:00,2025-11-05 12:25:00,2025-11-28 06:49:00,2026-03-26 20:23:00,NaN
updated_at,290835,NaN,NaN,NaN,2025-10-25 12:07:41.910774272,2025-02-19 07:00:00,2025-09-15 19:34:00,2025-11-19 13:03:00,2025-12-06 18:04:30,2026-03-27 01:49:00,NaN
access_finished_at,290741,NaN,NaN,NaN,2026-03-25 09:34:34.592850688,2025-02-16 00:00:00,2026-03-10 00:00:00,2026-05-06 00:00:00,2026-05-29 00:00:00,2026-09-26 00:00:00,NaN
wk_points,205925.0,<NA>,<NA>,<NA>,52.373169,0.0,28.0,52.299999,61.330002,492.079987,45.953308
wk_max_points,290710.0,<NA>,<NA>,<NA>,78.579185,0.0,4.0,63.0,79.0,1230.0,89.545662
wk_max_viewable_lessons,290710.0,<NA>,<NA>,<NA>,12.288298,0.0,5.0,14.0,14.0,80.0,7.222628
wk_max_task_count,290710.0,<NA>,<NA>,<NA>,82.034218,0.0,13.0,63.0,91.0,1230.0,91.185585


...

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
resource_type,852358,2,Lesson,619576,NaN,NaN,NaN,NaN,NaN,NaN,NaN
viewer_id,852358.0,<NA>,<NA>,<NA>,722081.52668,665744.0,704571.0,720112.0,741622.0,761599.0,22483.214357
segments_total,852358.0,<NA>,<NA>,<NA>,18.296854,1.0,8.0,10.0,37.0,190.0,17.155832
viewed_segments_count,852358.0,<NA>,<NA>,<NA>,11.93938,1.0,2.0,8.0,11.0,186.0,13.878893
started_at,852358,NaN,NaN,NaN,2025-11-24 09:50:05.723439872,2025-03-28 05:58:00,2025-11-01 12:35:00,2025-11-30 17:59:00,2025-12-12 05:27:00,2026-03-27 14:08:00,NaN
kind,852358,3,kinescope,619576,NaN,NaN,NaN,NaN,NaN,NaN,NaN



award_badges_df: 6 rows x 4 cols
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   title    6 non-null      category
 1   level    6 non-null      Int8    
 2   quota    6 non-null      Int16   
 3   special  6 non-null      boolean 
dtypes: Int16(1), Int8(1), boolean(1), category(1)
memory usage: 300.0 bytes


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
title,6,2,Я решаю,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
level,6.0,<NA>,<NA>,<NA>,2.666667,1.632993,1.0,1.25,2.5,3.75,5.0
quota,6.0,<NA>,<NA>,<NA>,113.5,192.799118,1.0,10.0,37.5,87.5,500.0
special,6,2,False,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN



user_award_badges_df: 252,843 rows x 3 cols
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252843 entries, 0 to 252842
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   award_badge_id  252843 non-null  Int8          
 1   user_id         252843 non-null  Int32         
 2   created_at      252843 non-null  datetime64[ns]
dtypes: Int32(1), Int8(1), datetime64[ns](1)
memory usage: 3.6 MB


,count,mean,min,25%,50%,75%,max,std
award_badge_id,252843.0,3.406205,1.0,2.0,3.0,4.0,6.0,1.186196
user_id,252843.0,719383.88025,663004.0,701568.0,721099.0,741223.0,761575.0,25648.447885
created_at,252843,2025-10-16 12:11:16.355762432,2023-10-03 12:53:00,2025-10-11 20:16:30,2025-11-17 07:44:00,2025-11-30 21:33:30,2026-03-27 17:18:00,NaN


По min/max значениям не нашла странных числел - напаример, отрицательных. 

user_trainings_df["attempts"]: min=max=1 -- колонка константная, бесполезна как фича.

## Проверка на дубли

In [22]:
# Полные дубли строк по всем импортированным таблицам.
duplicates_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "duplicate_rows": [int(df.duplicated().sum()) for df in dataframes.values()],
        "share_pct": [round(df.duplicated().mean() * 100, 4) for df in dataframes.values()],
    }
).sort_values(["duplicate_rows", "dataframe"], ascending=[False, True], ignore_index=True)

display(duplicates_df)


,dataframe,duplicate_rows,share_pct
0,wk_users_courses_actions_df,5491157,42.5367
1,user_answers_df,4951291,32.6254
2,user_access_histories_df,355372,53.2693
3,user_activity_histories_df,33312,1.0990
4,wk_media_view_sessions_df,22183,2.6025
5,lessons_df,1813,53.8142
6,award_badges_df,0,0.0000
7,user_award_badges_df,0,0.0000
8,user_lessons_df,0,0.0000
9,user_trainings_df,0,0.0000


### Sanity Checks

Следующая ячейка проверяет только базовые ограничения в основных таблицах: порядок дат, факт не превышает максимум и явные дубли в естественных ключах.


In [21]:
sanity_checks_df = pd.DataFrame(
    {
        "table": [
            "users_df",
            "users_df",
            "lessons_df",
            "user_access_histories_df",
            "user_lessons_df",
            "user_lessons_df",
            "user_answers_df",
            "user_trainings_df",
            "user_trainings_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "wk_users_courses_actions_df",
            "wk_media_view_sessions_df",
        ],
        "check": [
            "updated_at < created_at",
            "grade_changed_at < created_at",
            "lesson_number <= 0",
            "access_expired_at < access_started_at",
            "solved_tasks_count > wk_solved_task_count",
            "duplicate (user_id, users_course_id, lesson_id)",
            "attempts > max_attempts",
            "finished_at < started_at",
            "mark_saved_at < finished_at",
            "wk_points > wk_max_points",
            "wk_officially_started_at > access_finished_at",
            "wk_course_completed_at < created_at",
            "wk_course_completed_at < wk_officially_started_at",
            "duplicate (user_id, course_id)",
            "updated_at < created_at",
            "viewed_segments_count > segments_total",
        ],
        "bad_rows": [
            int((users_df["updated_at"] < users_df["created_at"]).sum()),
            int((users_df["grade_changed_at"] < users_df["created_at"]).sum()),
            int((lessons_df["lesson_number"] <= 0).sum()),
            int((user_access_histories_df["access_expired_at"] < user_access_histories_df["access_started_at"]).sum()),
            int((user_lessons_df["solved_tasks_count"] > user_lessons_df["wk_solved_task_count"]).sum()),
            int(user_lessons_df.duplicated(["user_id", "users_course_id", "lesson_id"]).sum()),
            int((user_answers_df["attempts"] > user_answers_df["max_attempts"]).sum()),
            int((user_trainings_df["finished_at"] < user_trainings_df["started_at"]).sum()),
            int((user_trainings_df["mark_saved_at"] < user_trainings_df["finished_at"]).sum()),
            int((users_courses_df["wk_points"] > users_courses_df["wk_max_points"]).sum()),
            int((users_courses_df["wk_officially_started_at"] > users_courses_df["access_finished_at"]).sum()),
            int((users_courses_df["wk_course_completed_at"] < users_courses_df["created_at"]).sum()),
            int((users_courses_df["wk_course_completed_at"] < users_courses_df["wk_officially_started_at"]).sum()),
            int(users_courses_df.duplicated(["user_id", "course_id"]).sum()),
            int((wk_users_courses_actions_df["updated_at"] < wk_users_courses_actions_df["created_at"]).sum()),
            int((wk_media_view_sessions_df["viewed_segments_count"] > wk_media_view_sessions_df["segments_total"]).sum()),
        ],
    }
)

table_rows = {name: df.shape[0] for name, df in dataframes.items()}
sanity_checks_df["share_pct"] = (sanity_checks_df["bad_rows"] / sanity_checks_df["table"].map(table_rows) * 100).round(4)

display(sanity_checks_df.sort_values(["bad_rows", "table"], ascending=[False, True], ignore_index=True))


,table,check,bad_rows,share_pct
0,user_access_histories_df,access_expired_at < access_started_at,2757,0.4133
1,user_answers_df,attempts > max_attempts,190,0.0013
2,users_courses_df,wk_points > wk_max_points,179,0.0615
3,user_lessons_df,solved_tasks_count > wk_solved_task_count,151,0.0049
4,user_trainings_df,mark_saved_at < finished_at,59,0.0138
5,users_courses_df,wk_officially_started_at > access_finished_at,2,0.0007
6,wk_users_courses_actions_df,updated_at < created_at,1,0.0000
7,lessons_df,lesson_number <= 0,0,0.0000
8,user_lessons_df,"duplicate (user_id, users_course_id, lesson_id)",0,0.0000
9,user_trainings_df,finished_at < started_at,0,0.0000


## Найденные проблемы со связностью

В users_courses.csv отсутствует users_course_id, поэтому эту таблицу нельзя напрямую связать с user_lessons.csv, wk_users_courses_actions.csv и user_access_histories.csv, где users_course_id есть.

В user_activity_histories.csv есть user_lesson_id, но в user_lessons.csv отсутствует user_lesson_id, поэтому user_activity_histories.csv нельзя напрямую связать ни с одной другой таблицей.

В lessons.csv отсутствует lesson_id, поэтому эту таблицу нельзя напрямую связать с user_lessons.csv и wk_users_courses_actions.csv, где lesson_id есть и по документации ссылается на урок из lessons.

В user_answers.csv отсутствует resource_id из документации, а также нет lesson_id, course_id и users_course_id, поэтому эту таблицу нельзя напрямую связать с lessons.csv, user_lessons.csv, users_courses.csv и wk_users_courses_actions.csv на уровне конкретного урока или курса.

В award_badges.csv отсутствует award_badge_id, поэтому эту таблицу нельзя напрямую связать с user_award_badges.csv, где award_badge_id есть.

В wk_media_view_sessions.csv отсутствуют lesson_id, course_id и users_course_id, поэтому эту таблицу нельзя напрямую связать с course-level блоком: users_courses.csv, user_lessons.csv, lessons.csv, wk_users_courses_actions.csv. Она связана только с пользователем через viewer_id.